In [1]:
import os
import numpy as np
import pandas as pd

PROCESSED_PATH = "processed_transactions.parquet"
RAW_PATH = "/Users/yuliia/Documents/Fraud-Detection/final_fraud_test_data_v4.csv"

def build_processed_df(raw_path: str) -> pd.DataFrame:
    df = pd.read_csv(raw_path)
    df["person_id"] = pd.to_numeric(df["person_id"], errors="coerce").astype("Int64")
    df = df[df["person_id"].notna()]

    date_cols = ["trn_date", "person_creation_date", "person_first_transaction"]
    for col in date_cols:
        df[col] = pd.to_datetime(df[col])

    df = df[df["person_first_transaction"] >= pd.Timestamp("2023-09-01", tz="UTC")]

    df["date"] = df["trn_date"].dt.date
    df["week"] = df["trn_date"].dt.to_period("W").dt.start_time
    df["month"] = df["trn_date"].dt.to_period("M").dt.start_time
    df["hour"] = df["trn_date"].dt.hour
    df["dow"] = df["trn_date"].dt.day_name()

    df["trn_id"] = df["trn_id"].astype("string")
    df["place_id"] = df["place_id"].astype("string")
    df["partner_id"] = df["partner_id"].astype("string")
    df["trn_reg_user_code"] = df["trn_reg_user_code"].astype("string")

    df["waiter_id"] = (
        df["place_id"].astype("string").fillna("")
        + "_"
        + df["trn_reg_user_code"].astype("string").fillna("")
    )
    df['waiter_week'] = df['waiter_id'].astype(str) + '_' + df['week'].astype(str)
    df['waiter_month'] = df['waiter_id'].astype(str) + '_' + df['month'].astype(str)

    # транзакції, де використовувались бофони, але вони не врахувались в discount_amount
    error_trn = df[(df["discount_amount"] == 0) & (df["bonusses_used"] != 0)]["trn_id"].values
    df = df[~df["trn_id"].isin(error_trn)]
    # забрати транзакції з відємним чеком
    df = df[df["total_amount"] >= 0]
    df["gross_amount"] = df["total_amount"] + df["discount_amount"]

    # зробити bonusses_used додатним
    df["bonusses_used"] = (-1) * df["bonusses_used"]

    # ceil discount_amount
    df["discount_amount"] = np.ceil(df["discount_amount"])

    # прибираю проблемні транзакції, де сума бонусів не врахувалась в суму дисконту
    df = df[df["discount_amount"] >= df["bonusses_used"]]
    df["promo_used"] = df["discount_amount"] - df["bonusses_used"]

    df["is_first_trn"] = df["trn_count_before"] == 0
    df["bonus_used_flag"] = df["bonusses_used"] > 0

    df = df.sort_values(["person_id", "trn_date"])

    df["time_since_prev_trn_hours"] = (
        df.groupby("person_id")["trn_date"]
          .diff()
          .dt.total_seconds() / 3600
    )

    # відфільтровую заклади з малою к-тю офіціантів (поки до 4ох)
    waiters_per_place = df.groupby('place_id').agg(
        num_of_waiters = ('waiter_id', 'nunique')
    ).reset_index()

    few_waiters_per_place = waiters_per_place[waiters_per_place['num_of_waiters'] <3]['place_id']
    few_waiters_per_place

    df = df[~df['place_id'].isin(few_waiters_per_place)]
    df['balance_after_all_processing'] = (
        df['balance_before_all_processing']
        - df['bonusses_used']
        + df['bonusses_accum']
    ).fillna(0)

    df['other_processings'] = (
        df['balance_before_all_processing']
        - df.groupby('person_id')['balance_after_all_processing'].shift(1)
    ).fillna(0)

    df['gross_amount_90'] = df['place_id'].map(df.groupby('place_id')['gross_amount'].quantile(0.9))
    df['gross_amount_95'] = df['place_id'].map(df.groupby('place_id')['gross_amount'].quantile(0.95))

    df['exp_90'] = df['gross_amount'] >= df['gross_amount_90']
    df['exp_95'] = df['gross_amount'] >= df['gross_amount_95']

    return df

if os.path.exists(PROCESSED_PATH):
    df = pd.read_parquet(PROCESSED_PATH)
else:
    df = build_processed_df(RAW_PATH)
    df.to_parquet(PROCESSED_PATH, index=False)

/var/folders/_c/vc94t3xj39q4v2bl8kw61d5m0000gp/T/ipykernel_97804/3657879380.py:9: DtypeWarning: Columns (2,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(raw_path)
/var/folders/_c/vc94t3xj39q4v2bl8kw61d5m0000gp/T/ipykernel_97804/3657879380.py:20: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["week"] = df["trn_date"].dt.to_period("W").dt.start_time
/var/folders/_c/vc94t3xj39q4v2bl8kw61d5m0000gp/T/ipykernel_97804/3657879380.py:21: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["month"] = df["trn_date"].dt.to_period("M").dt.start_time
